# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [1]:
# Uncomment in a fresh Kaggle notebook environment.
# %pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

In [2]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.11.0+cu130
CUDA available: True


In [ ]:
# Core training config.
from pathlib import Path

default_output_dir = (
    "/kaggle/working/qwen3b_svg_lora"
    if Path("/kaggle/working").exists()
    else str(Path("./artifacts/qwen3b_svg_lora").resolve())
)

CONFIG = {
    "model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "max_seq_length": 1024,
    "lora_r": 8,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 100,
    "eval_size": 0.02,
    "output_dir": default_output_dir,
}

print(f"Using output_dir: {CONFIG['output_dir']}")
CONFIG

Using output_dir: /home/anthonylamelas/Coding/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora


{'model_name': 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
 'max_seq_length': 1024,
 'lora_r': 8,
 'lora_alpha': 16,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 16,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 100,
 'eval_size': 0.02,
 'output_dir': '/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora'}

In [4]:
# Competition training source (VS Code + Colab extension friendly)
from pathlib import Path

IS_COLAB = False
try:
    from google.colab import drive
    IS_COLAB = True
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

if IS_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/DL_M1')
    TRAIN_CSV_PATH = str(DATA_DIR / 'train.csv')
    TEST_CSV_PATH = str(DATA_DIR / 'test.csv')
else:
    TRAIN_CSV_PATH = 'data/train.csv'
    TEST_CSV_PATH = 'data/test.csv'

if not Path(TRAIN_CSV_PATH).exists():
    raise FileNotFoundError(f'Train file not found: {TRAIN_CSV_PATH}')
if not Path(TEST_CSV_PATH).exists():
    raise FileNotFoundError(f'Test file not found: {TEST_CSV_PATH}')

print(f'Using train file: {TRAIN_CSV_PATH}')
print(f'Using test file: {TEST_CSV_PATH}')

Using train file: data/train.csv
Using test file: data/test.csv


In [5]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
required_cols = {"prompt", "svg"}
missing = required_cols - set(train_df.columns)
if missing:
    raise ValueError(f"Missing required columns in {TRAIN_CSV_PATH}: {sorted(missing)}")

train_df = train_df.dropna(subset=["prompt", "svg"]).copy()
train_df["prompt"] = train_df["prompt"].astype(str).str.strip()
train_df["svg"] = train_df["svg"].astype(str).str.strip()
train_df = train_df[(train_df["prompt"] != "") & (train_df["svg"].str.lower().str.startswith("<svg"))]

if CONFIG["max_train_samples_per_source"] and len(train_df) > CONFIG["max_train_samples_per_source"]:
    train_df = train_df.sample(CONFIG["max_train_samples_per_source"], random_state=SEED)

if len(train_df) == 0:
    raise RuntimeError("No usable rows found in train.csv after filtering.")

train_raw = Dataset.from_pandas(train_df[["prompt", "svg"]], preserve_index=False)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Train rows: 98
Eval rows: 2


{'prompt': 'The image contains a solid black circle with a diagonal black line inside it, placed centrally within the frame.',
 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#181818" fill-opacity="1.0"  filling="0" d="M139.52499389648438 148.3625030517578 L51.631248474121094 60.46875 C54.275001525878906 57.23749923706055 57.243751525878906 54.281253814697266 60.474998474121094 51.631248474121094 L148.3625030517578 139.52499389648438 C145.71875 142.75625610351562 142.75625610351562 145.71249389648438 139.52499389648438 148.3625030517578 M100.0 12.5 C51.66875076293945 12.5 12.5 51.67499923706055 12.5 100.0 C12.5 148.3249969482422 51.66875076293945 187.5 100.0 187.5 C148.3249969482422 187.5 187.5 148.3249969482422 187.5 100.0 C187.5 51.67499923706055 148.3249969482422 12.5 100.0 12.5"></path></svg>'}

In [6]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

Map: 100%|██████████| 2/2 [00:00<00:00, 828.75 examples/s]

<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element.<|im_end|>
<|im_start|>user
The image contains a solid black circle with a diagonal black line inside it, placed centrally within the frame.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="2


In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 9.636 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 1391.60it/s]


unsloth/qwen2.5-3b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
from pathlib import Path
import subprocess
import sysconfig

from transformers import TrainingArguments
from trl import SFTTrainer

python_include_dir = Path(sysconfig.get_paths().get("include", ""))
python_h = python_include_dir / "Python.h"
if not python_h.exists():
    raise EnvironmentError(
        "Missing Python development headers (Python.h). "
        f"Expected at: {python_h}. "
        "Install system package python3.12-dev (or python3-dev), restart the kernel, and rerun from Cell 4."
    )

if not torch.cuda.is_available():
    raise EnvironmentError(
        "CUDA GPU is not available in this kernel, but Unsloth LoRA training requires CUDA/Triton."
    )

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    evaluation_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
 )

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
 )

try:
    train_result = trainer.train()
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        "Triton CUDA build failed during training. "
        "Most common local fix: install Python dev headers (python3.12-dev) and ensure NVIDIA driver libraries are visible, "
        "then restart kernel and rerun from Cell 4. "
        f"Original command: {exc.cmd}"
    ) from exc

train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Tokenizing ["text"] (num_proc=7): 100%|██████████| 98/98 [00:07<00:00, 13.59 examples/s]
num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
[datasets.arrow_dataset|WARNING]num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
Unsloth: Tokenizing ["text"] (num_proc=2): 100%|██████████| 2/2 [00:02<00:00,  1.25s/ examples]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 98 | Num Epochs = 1 | Total steps = 7
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /

Step,Training Loss


KeyboardInterrupt: 

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

Saved adapter + tokenizer to: /home/anthonylamelas/Coding/Deep_Learning_Midterm_1/artifacts/qwen2b_svg_lora


In [ ]:
FastLanguageModel.for_inference(model)
model.eval()

SVG_REGEX = re.compile(r"<svg[\\s\\S]*?</svg>", flags=re.IGNORECASE)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=192):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.02,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg


def generate_svg_batch(prompts, max_new_tokens=192):
    chat_texts = [
        (
            "<|im_start|>system\n"
            f"{SYSTEM_PROMPT}<|im_end|>\n"
            "<|im_start|>user\n"
            f"{prompt}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        for prompt in prompts
    ]
    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_seq_length"],
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.02,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded_batch = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    svgs = []
    for prompt, decoded in zip(prompts, decoded_batch):
        svg = extract_svg(decoded)
        if not is_valid_svg(svg):
            svg = fallback_svg(prompt)
        svgs.append(svg)
    return svgs


test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.m

<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="black"/></svg>
Valid SVG: True


In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv"
# SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Save locally
TEST_PROMPTS_PATH = TEST_CSV_PATH  
SUBMISSION_PATH = "./submission.csv"  

test_df = pd.read_csv(TEST_PROMPTS_PATH)

batch_size = 16
rows = []
invalid_count = 0
t0 = time.time()

for start in range(0, len(test_df), batch_size):
    batch_df = test_df.iloc[start:start + batch_size]
    prompts = batch_df["prompt"].tolist()
    ids = batch_df["id"].tolist()

    svgs = generate_svg_batch(prompts, max_new_tokens=192)

    for sample_id, prompt, svg in zip(ids, prompts, svgs):
        if not is_valid_svg(svg):
            invalid_count += 1
            svg = fallback_svg(prompt)
        rows.append({"id": sample_id, "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.m

Saved: ./submission.csv
Rows: 1000
Invalid/fallback count: 0
Runtime (minutes): 186.67


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."


## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.